### Define expected tables and keys

In [0]:
from pyspark.sql import functions as F

CATALOG = "olist_mlops"
SCHEMA = "silver"

# Expected rows and candidate primary keys.
# None means no individual key has been assigned.

EXPECTED_TABLES = {
    "orders_clean": (99441, ["order_id"]),
    "customers_clean": (99441, ["customer_id"]),
    "order_items_clean": (
        112650, ["order_id", "order_item_id"]
    ),
    "payments_clean": (
        103886, ["order_id", "payment_sequential"]
    ),
    "reviews_clean": (
        99224, ["review_id", "order_id"]
    ),
    "products_clean": (32951, ["product_id"]),
    "sellers_clean": (3095, ["seller_id"]),
    "category_translation_clean": (
        71, ["product_category_name"]
    ),
    "geolocation_clean": (738332, None),
    "geolocation_zip_lookup": (
        19015, ["zip_prefix_5d"]
    )
}

print("Silver tables configured:", len(EXPECTED_TABLES))

### Implement row-count and key validation

In [0]:
validation_results = []

for table_name, (expected_rows, key_columns) in EXPECTED_TABLES.items():

    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"

    print(f"\nValidating: {full_name}")

    df = spark.table(full_name)

    actual_rows = df.count()

    null_key_rows = None
    duplicate_key_extras = None

    if key_columns:

        key_df = df.select(*key_columns)

        null_condition = F.lit(False)

        for column in key_columns:
            null_condition = (
                null_condition | F.col(column).isNull()
            )

        null_key_rows = key_df.filter(
            null_condition
        ).count()

        unique_keys = key_df.distinct().count()

        duplicate_key_extras = actual_rows - unique_keys

    passed = (
        actual_rows == expected_rows
        and (
            key_columns is None
            or (
                null_key_rows == 0
                and duplicate_key_extras == 0
            )
        )
    )

    validation_results.append({
        "table_name": table_name,
        "expected_rows": expected_rows,
        "actual_rows": actual_rows,
        "null_key_rows": null_key_rows,
        "duplicate_key_extras": duplicate_key_extras,
        "status": "PASS" if passed else "FAIL"
    })

    print("Expected rows:", expected_rows)
    print("Actual rows:", actual_rows)
    print("Status:", "PASS" if passed else "FAIL")

    if not passed:
        raise AssertionError(
            f"Silver validation failed: {table_name}"
        )

print("\nALL SILVER TABLE VALIDATIONS PASSED")

### Display the validation summary

In [0]:
summary_df = spark.createDataFrame(validation_results)

display(
    summary_df.orderBy("table_name")
)

assert all(
    result["status"] == "PASS"
    for result in validation_results
)

print("SILVER TABLE VALIDATION: 10/10 PASSED")

### Validate relationships across Silver tables
- Foreign-key testing

In [0]:
RELATIONSHIPS = [
    ("orders_clean", "customer_id",
     "customers_clean", "customer_id"),

    ("order_items_clean", "order_id",
     "orders_clean", "order_id"),

    ("order_items_clean", "product_id",
     "products_clean", "product_id"),

    ("order_items_clean", "seller_id",
     "sellers_clean", "seller_id"),

    ("payments_clean", "order_id",
     "orders_clean", "order_id"),

    ("reviews_clean", "order_id",
     "orders_clean", "order_id")
]

fk_results = []

for child_table, child_key, parent_table, parent_key in RELATIONSHIPS:

    child = (
        spark.table(f"olist_mlops.silver.{child_table}")
        .select(F.col(child_key).alias("key"))
    )

    parent = (
        spark.table(f"olist_mlops.silver.{parent_table}")
        .select(F.col(parent_key).alias("key"))
        .distinct()
    )

    # NULL foreign keys are checked separately.
    null_keys = child.filter(
        F.col("key").isNull()
    ).count()

    orphan_rows = (
        child.filter(F.col("key").isNotNull())
        .join(parent, on="key", how="left_anti")
        .count()
    )

    fk_results.append({
        "relationship": f"{child_table} -> {parent_table}",
        "null_keys": null_keys,
        "orphan_rows": orphan_rows,
        "status": (
            "PASS"
            if null_keys == 0 and orphan_rows == 0
            else "FAIL"
        )
    })

    print(
        f"{child_table} -> {parent_table}: "
        f"{orphan_rows} orphans, {null_keys} null keys"
    )

    assert orphan_rows == 0
    assert null_keys == 0

display(spark.createDataFrame(fk_results))

print("SILVER FOREIGN-KEY VALIDATION: 6/6 PASSED")

### Validate lookup coverage and business rules

In [0]:
%sql

SELECT
    location_status,
    COUNT(*) AS seller_count
FROM olist_mlops.silver.sellers_clean
GROUP BY location_status
ORDER BY location_status;

In [0]:
%sql

SELECT
    category_translation_status,
    COUNT(*) AS products
FROM olist_mlops.silver.products_clean
GROUP BY category_translation_status
ORDER BY category_translation_status;

In [0]:
%sql

SELECT
    CASE
        WHEN product_weight_g IS NULL THEN 'MISSING'
        WHEN product_weight_g <= 0 THEN 'NON_POSITIVE'
        ELSE 'VALID'
    END AS product_weight_quality,
    COUNT(*) AS products
FROM olist_mlops.silver.products_clean
GROUP BY product_weight_quality;

In [0]:
%sql

SELECT
    payment_value_quality,
    installment_quality,
    COUNT(*) AS payment_records
FROM olist_mlops.silver.payments_clean
GROUP BY
    payment_value_quality,
    installment_quality
ORDER BY
    payment_value_quality,
    installment_quality;